In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from functools import reduce


In [3]:
spark = SparkSession.builder \
    .appName("India Weather Analysis") \
    .getOrCreate()

In [5]:
df = spark.read.csv(r"D:\BDA 17\ABD\weather\Weather Data in India from 1901 to 2017.csv", header =True, inferSchema = True)

In [6]:
df.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- JAN: double (nullable = true)
 |-- FEB: double (nullable = true)
 |-- MAR: double (nullable = true)
 |-- APR: double (nullable = true)
 |-- MAY: double (nullable = true)
 |-- JUN: double (nullable = true)
 |-- JUL: double (nullable = true)
 |-- AUG: double (nullable = true)
 |-- SEP: double (nullable = true)
 |-- OCT: double (nullable = true)
 |-- NOV: double (nullable = true)
 |-- DEC: double (nullable = true)



In [7]:
month_cols = [col for col in df.columns if col != "Year"]

In [8]:
# 1. Add column average temperature for each year
avg_expr = reduce(lambda a, b: a + b, [col(c) for c in month_cols])

df = df.withColumn(
    "Year_Avg_Temp",
    round(avg_expr / len(month_cols), 2)
)

print("Average Temperature for Each Year")
df.select("Year", "Year_Avg_Temp").show()


Average Temperature for Each Year
+----+-------------+
|Year|Year_Avg_Temp|
+----+-------------+
|1901|       168.59|
|1902|       168.76|
|1903|       168.52|
|1904|       168.48|
|1905|       168.63|
|1906|       168.86|
|1907|       168.85|
|1908|       168.88|
|1909|        168.8|
|1910|       168.86|
|1911|       169.11|
|1912|       169.33|
|1913|        169.2|
|1914|       169.42|
|1915|       169.73|
|1916|       169.61|
|1917|       169.22|
|1918|       169.53|
|1919|       169.78|
|1920|       169.76|
+----+-------------+
only showing top 20 rows



In [10]:
# 2. Add row average temperature of each month
import builtins

monthly_avg = []

for month in month_cols:
    avg_temp = df.select(avg(col(month))).collect()[0][0]
    monthly_avg.append((month, builtins.round(avg_temp, 2)))

monthly_avg_df = spark.createDataFrame(
    monthly_avg,
    ["Month", "Average_Temperature"]
)

monthly_avg_df.show()

+-----+-------------------+
|Month|Average_Temperature|
+-----+-------------------+
| YEAR|             1959.0|
|  JAN|              18.42|
|  FEB|              20.14|
|  MAR|              23.43|
|  APR|              26.51|
|  MAY|              28.39|
|  JUN|               28.3|
|  JUL|              27.37|
|  AUG|              26.94|
|  SEP|              26.34|
|  OCT|              24.74|
|  NOV|              21.77|
|  DEC|              19.17|
+-----+-------------------+



In [12]:
# 3. Add columns Min and Max for each year
df = df.withColumn(
    "Min_Temp",
    least(*[col(c) for c in month_cols])
)

df = df.withColumn(
    "Max_Temp",
    greatest(*[col(c) for c in month_cols])
)

print("Year-wise Min and Max Temperatures")
df.select("Year", "Min_Temp", "Max_Temp").show()

Year-wise Min and Max Temperatures
+----+--------+--------+
|Year|Min_Temp|Max_Temp|
+----+--------+--------+
|1901|   17.99|  1901.0|
|1902|   18.78|  1902.0|
|1903|   18.29|  1903.0|
|1904|   17.77|  1904.0|
|1905|    17.4|  1905.0|
|1906|    17.5|  1906.0|
|1907|   18.46|  1907.0|
|1908|   18.15|  1908.0|
|1909|   17.79|  1909.0|
|1910|   18.05|  1910.0|
|1911|   18.52|  1911.0|
|1912|   18.44|  1912.0|
|1913|    18.2|  1913.0|
|1914|   18.73|  1914.0|
|1915|   17.93|  1915.0|
|1916|   18.17|  1916.0|
|1917|   18.16|  1917.0|
|1918|   17.25|  1918.0|
|1919|   18.58|  1919.0|
|1920|   18.27|  1920.0|
+----+--------+--------+
only showing top 20 rows



In [14]:
# 4. Create dataframe 'decade'
#    Decade 1910 => 1901 to 1910

df_decade = df.withColumn(
    "Decade",
    ((col("Year") - 1) / 10).cast("int") * 10 + 10
)

decade = df_decade.groupBy("Decade").agg(
    *[
        round(avg(col(month)), 2).alias(month)
        for month in month_cols
    ]
).orderBy("Decade")

print("Average Monthly Temperature by Decade")
decade.show(truncate=False)

Average Monthly Temperature by Decade
+------+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|Decade|YEAR  |JAN  |FEB  |MAR  |APR  |MAY  |JUN  |JUL  |AUG  |SEP  |OCT  |NOV  |DEC  |
+------+------+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|1910  |1905.5|18.15|19.38|22.83|26.28|28.2 |28.16|27.27|26.75|26.04|24.67|21.5 |18.66|
|1920  |1915.5|18.32|19.62|22.8 |26.01|28.09|28.04|27.37|26.82|26.12|24.5 |21.39|18.52|
|1930  |1925.5|18.22|19.9 |23.5 |26.48|28.51|28.24|27.28|26.79|26.08|24.35|21.26|18.88|
|1940  |1935.5|18.2 |19.97|22.95|26.17|28.39|28.16|27.23|26.77|26.26|24.53|21.48|18.97|
|1950  |1945.5|18.17|19.66|23.24|26.38|28.55|28.35|27.31|26.95|26.21|24.63|21.49|18.94|
|1960  |1955.5|18.45|20.23|23.43|26.41|28.3 |28.28|27.19|26.83|26.33|24.56|21.53|19.32|
|1970  |1965.5|18.13|20.14|23.47|26.3 |28.0 |28.22|27.26|26.83|26.24|24.58|21.63|18.98|
|1980  |1975.5|18.3 |19.87|23.26|26.68|28.14|28.19|27.2 |26.73|26.21|24.75|21.89|1

In [15]:
# 5. Hottest Year (Based on Average Temperature)
hottest_year = df.orderBy(
    col("Year_Avg_Temp").desc()
).limit(1)

print("Hottest Year")
hottest_year.select(
    "Year",
    "Year_Avg_Temp"
).show()

Hottest Year
+----+-------------+
|Year|Year_Avg_Temp|
+----+-------------+
|2016|        179.5|
+----+-------------+



In [16]:
# 6. Coldest Year (Based on Average Temperature)
coldest_year = df.orderBy(
    col("Year_Avg_Temp").asc()
).limit(1)

print("Coldest Year")
coldest_year.select(
    "Year",
    "Year_Avg_Temp"
).show()

Coldest Year
+----+-------------+
|Year|Year_Avg_Temp|
+----+-------------+
|1904|       168.48|
+----+-------------+



In [17]:
# 7. Years Recording Absolute Minimum and Maximum
#    Temperatures (Not based on average)

max_temp_value = df.agg(
    max("Max_Temp")
).collect()[0][0]

min_temp_value = df.agg(
    min("Min_Temp")
).collect()[0][0]

year_max_temp = df.filter(
    col("Max_Temp") == max_temp_value
)

year_min_temp = df.filter(
    col("Min_Temp") == min_temp_value
)

print("Year(s) with Maximum Recorded Temperature")
year_max_temp.select(
    "Year",
    "Max_Temp"
).show()

print("Year(s) with Minimum Recorded Temperature")
year_min_temp.select(
    "Year",
    "Min_Temp"
).show()

Year(s) with Maximum Recorded Temperature
+----+--------+
|Year|Max_Temp|
+----+--------+
|2017|  2017.0|
+----+--------+

Year(s) with Minimum Recorded Temperature
+----+--------+
|Year|Min_Temp|
+----+--------+
|1918|   17.25|
+----+--------+



In [ ]:
#8.How much raise is observed in min and max temperature for each month?
raise_data = []

for month in month_cols:

    min_temp = float(df.select(min(col(month))).first()[0])
    max_temp = float(df.select(max(col(month))).first()[0])

    increase = builtins.round(max_temp - min_temp, 2)

    raise_data.append(
        (month, min_temp, max_temp, increase)
    )

raise_df = spark.createDataFrame(
    raise_data,
    ["Month", "Min_Temp", "Max_Temp", "Temperature_Raise"]
)

raise_df.show()

+-----+--------+--------+-----------------+
|Month|Min_Temp|Max_Temp|Temperature_Raise|
+-----+--------+--------+-----------------+
| YEAR|  1901.0|  2017.0|            116.0|
|  JAN|   17.25|   20.92|             3.67|
|  FEB|   17.79|   23.58|             5.79|
|  MAR|   21.78|   26.61|             4.83|
|  APR|   24.84|   29.56|             4.72|
|  MAY|   26.97|   30.78|             3.81|
|  JUN|   27.33|   29.88|             2.55|
|  JUL|   26.48|   28.47|             1.99|
|  AUG|   26.21|   28.17|             1.96|
|  SEP|   25.47|   28.11|             2.64|
|  OCT|   23.52|   27.24|             3.72|
|  NOV|   20.59|   23.92|             3.33|
|  DEC|   17.98|   21.89|             3.91|
+-----+--------+--------+-----------------+

